In [1]:
import numpy as np
from scipy.spatial.distance import pdist, squareform

DATA_PATH = (
    "/home/maria/SelfStudyThesis/data/"
    "allen_natural_scenes_processed_data.npy"
)

dat = np.load(DATA_PATH, allow_pickle=True).item()

X = np.asarray(dat["X"], dtype=np.float64).T

labels = np.asarray(
    dat["stimulus_metadata"]["label"]
)

# Animal = 1
y = (labels == 0).astype(int)

# Standardize each neuron's responses
mu = X.mean(axis=0)
sd = X.std(axis=0)

Xz = (X - mu) / np.maximum(sd, 1e-8)

# Pairwise squared Euclidean distances
D2 = squareform(pdist(Xz, metric="sqeuclidean"))

# Median bandwidth heuristic
upper = D2[np.triu_indices_from(D2, k=1)]
bandwidth = np.median(upper)

# Gaussian similarity
K = np.exp(-D2 / max(bandwidth, 1e-12))

# Exclude self-transitions
np.fill_diagonal(K, 0.0)

# Construct Markov transition matrix
P = K / K.sum(axis=1, keepdims=True)

print("Shape:", P.shape)
print("Row sums:", P.sum(axis=1))

Shape: (118, 118)
Row sums: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [2]:
animal = y == 1
inanimate = y == 0

R_animal = P[np.ix_(animal, animal)].sum(axis=1).mean()
R_inanimate = P[np.ix_(inanimate, inanimate)].sum(axis=1).mean()

print("Animal retention:", R_animal)
print("Inanimate retention:", R_inanimate)

Animal retention: 0.46628818955459234
Inanimate retention: 0.5288366924032327
